In [ ]:
! pip install langchain langchain-openai openai langchain_community faiss-gpu

In [9]:
import os

os.environ["OPENAI_API_KEY"] = "API_key"

In [4]:
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [10]:

doc1 = Document(
    page_content="""Python is a popular programming language used for web development.
Redis caching improves API performance by storing frequently accessed data in memory.
Python also supports automation, scripting, and data analysis.
Redis reduces repeated database queries and decreases API response time."""
)

doc2 = Document(
    page_content="""AWS provides cloud services such as EC2, S3, and Lambda.
Asynchronous programming allows APIs to handle multiple I/O operations concurrently.
Amazon S3 is commonly used for storing images, files, and backups.
AsyncIO improves API concurrency by avoiding blocking during I/O operations."""
)

doc3 = Document(
    page_content="""SQL is used to retrieve and manipulate relational database records.
Database connection pooling reuses existing connections to reduce overhead.
SQL supports operations such as JOIN, GROUP BY, and ORDER BY.
Connection pooling can improve API performance when database access is frequent."""
)

doc4 = Document(
    page_content="""Docker packages applications and dependencies into portable containers.
Load balancing distributes incoming API requests across multiple servers.
Docker images help maintain consistent deployment environments.
Horizontal scaling allows applications to handle increased user traffic."""
)

doc5 = Document(
    page_content="""Machine learning models learn patterns from historical datasets.
Background task queues move time-consuming work outside the API request cycle.
Scikit-learn provides algorithms for classification and regression.
Celery workers process background jobs without blocking API responses."""
)

documents = [doc1, doc2, doc3, doc4, doc5]

In [11]:
vector_db = FAISS.from_documents(
    documents = documents,
    embedding = OpenAIEmbeddings()
)

In [12]:
base_retriever = vector_db.as_retriever(kwargs={"k":3})

In [19]:
compressor = LLMChainExtractor.from_llm(
    llm=ChatOpenAI(model="gpt-5.2")
)

In [22]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [26]:
compression_docs = compression_retriever.invoke("What techniques can improve API performance and handle more concurrent users?")

In [27]:
for i,doc in enumerate(compression_docs):
  print(f"Result {i} : {doc.page_content}")


Result 0 : Background task queues move time-consuming work outside the API request cycle.
Celery workers process background jobs without blocking API responses.
Result 1 : Asynchronous programming allows APIs to handle multiple I/O operations concurrently.
AsyncIO improves API concurrency by avoiding blocking during I/O operations.
Result 2 : Redis caching improves API performance by storing frequently accessed data in memory.
Redis reduces repeated database queries and decreases API response time.
Result 3 : Database connection pooling reuses existing connections to reduce overhead.  
Connection pooling can improve API performance when database access is frequent.
